In [3]:
# =====================================================================
# QFrFT-PQCNet : Hybrid Quantum-Classical Seismic Denoising Framework
# =====================================================================
# =====================================================================
# IMPLEMENTATION OVERVIEW
# =====================================================================
#
# This implementation follows the methodology proposed in:
#
# Section 3:
# Proposed Hybrid Quantum-Classical Framework
# Main Components:
# 1. Quantum Fractional Fourier Transform (QFrFT)
# 2. Fractional-domain adaptive filtering
# 3. Parameterized Quantum Circuits (PQC)
# 4. Cascaded hybrid QFrFT-PQC architecture
# 5. Composite objective optimization
# =====================================================================
#!pip install pennylane
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pennylane as qml

# =====================================================================
# REPRODUCIBILITY
# =====================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =====================================================================
# DEVICE CONFIGURATION
# =====================================================================
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device:", DEVICE)
# =====================================================================
# MODEL PARAMETERS
# =====================================================================
N = 256
N_QUBITS = 8
PQC_LAYERS = 2
BATCH_SIZE = 8
EPOCHS = 100
LEARNING_RATE = 3e-3
# =====================================================================
# COMPOSITE LOSS WEIGHTS
# =====================================================================
# Total objective:
# L = λ1Ltime + λ2Lfrac + λ3Lcorr
# Corresponds to:Eq. (29)–(32)
# =====================================================================
LAMBDA_TIME = 1.0
LAMBDA_FRAC = 0.15
LAMBDA_CORR = 0.05
# =====================================================================
# SYNTHETIC SEISMIC DATASET
# =====================================================================
# Synthetic seismic generation including:
# - Ricker wavelets
# - Chirp events
# - Gaussian noise
# - Burst noise
# - Colored noise
# =====================================================================
class SyntheticSeismicDataset(Dataset):

    def __init__(
        self,
        n_samples=800,
        snr_range=(0, 15)
    ):

        self.clean = []
        self.noisy = []

        for _ in range(n_samples):

            clean = self.generate_clean_signal()

            noisy = self.add_noise(
                clean,
                snr_range
            )

            clean = self.normalize(clean)

            noisy = self.normalize(noisy)

            self.clean.append(
                clean.astype(np.float32)
            )

            self.noisy.append(
                noisy.astype(np.float32)
            )

        self.clean = np.array(self.clean)
        self.noisy = np.array(self.noisy)

    # ================================================================

    def ricker(self, points, width):

        t = np.linspace(-1, 1, points)

        wavelet = (
            1
            -
            2 * (np.pi**2) * (width**2) * (t**2)
        )

        wavelet *= np.exp(
            -(np.pi**2) * (width**2) * (t**2)
        )

        return wavelet

    # ================================================================

    def generate_clean_signal(self):

        signal = np.zeros(N)

        n_events = np.random.randint(3, 7)

        for _ in range(n_events):

            center = np.random.randint(
                20,
                N - 20
            )

            width = np.random.uniform(4, 12)

            amplitude = np.random.uniform(
                0.5,
                1.5
            )

            wavelet = amplitude * self.ricker(
                N,
                width
            )

            shift = center - N // 2

            wavelet = np.roll(
                wavelet,
                shift
            )

            signal += wavelet

        # ============================================================
        # Chirp Component
        # ============================================================

        t = np.linspace(0, 1, N)

        f0 = np.random.uniform(5, 15)

        f1 = np.random.uniform(30, 60)

        chirp = np.sin(
            2 * np.pi * (
                f0 * t
                +
                0.5 * (f1 - f0) * t**2
            )
        )

        chirp *= np.exp(-3 * t)

        signal += 0.3 * chirp

        return signal

    # ================================================================

    def add_noise(self, clean, snr_range):

        snr_db = np.random.uniform(*snr_range)

        signal_power = np.mean(clean**2)

        noise_power = (
            signal_power
            /
            (10**(snr_db / 10))
        )

        gaussian_noise = np.sqrt(
            noise_power
        ) * np.random.randn(N)

        # ============================================================
        # Burst Noise
        # ============================================================

        burst = np.zeros(N)

        if np.random.rand() > 0.5:

            start = np.random.randint(
                30,
                N - 50
            )

            width = np.random.randint(
                10,
                30
            )

            burst_signal = np.random.randn(width)

            burst_signal *= np.hanning(width)

            burst[start:start+width] += burst_signal

        # ============================================================
        # Colored Noise
        # ============================================================

        colored = np.cumsum(
            np.random.randn(N)
        )

        colored = (
            colored
            /
            np.max(np.abs(colored))
        )

        total_noise = (
            gaussian_noise
            +
            0.2 * burst
            +
            0.1 * colored
        )

        return clean + total_noise

    # ================================================================

    def normalize(self, x):

        x = (
            x - np.mean(x)
        ) / (
            np.std(x) + 1e-8
        )

        return x

    # ================================================================

    def __len__(self):

        return len(self.clean)

    # ================================================================

    def __getitem__(self, idx):

        noisy = torch.tensor(self.noisy[idx])

        clean = torch.tensor(self.clean[idx])

        return noisy, clean

# =====================================================================
# DATASET CREATION
# =====================================================================
train_dataset = SyntheticSeismicDataset(
    n_samples=800
)

val_dataset = SyntheticSeismicDataset(
    n_samples=200
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# =====================================================================
# Quantum Fractional Fourier Transform (QFrFT)
# =====================================================================
# Implements Section 3.2 of the paper:
# QFrFTα{|ψ>} = Fα|ψ>
# where:
# - α is the learnable fractional transform order
# - Fα is the fractional Fourier operator
# The transform performs adaptive rotation in the
# time-frequency plane to obtain compact energy
# concentration of coherent seismic events.
# Corresponds to:Eq. (8)–(13)
# =====================================================================

class LearnableQFrFT(nn.Module):

    def __init__(self, N):

        super().__init__()

        self.N = N

        # Prevent alpha collapse

        self.alpha_raw = nn.Parameter(
            torch.tensor(0.3)
        )

    # ================================================================

    @property
    def alpha(self):

        # constrain alpha between 0.05 and 0.95

        return (
            0.05
            +
            0.90 * torch.sigmoid(self.alpha_raw)
        )

    # ================================================================

    def forward(self, x):

        X = torch.fft.fft(
            x,
            dim=1
        )

        freqs = torch.linspace(
            0,
            1,
            self.N,
            device=x.device
        )

        phase = torch.exp(
            -1j
            *
            np.pi
            *
            self.alpha
            *
            freqs
        )

        X_frac = X * phase

        return X_frac

    # ================================================================

    def inverse(self, X_frac):

        freqs = torch.linspace(
            0,
            1,
            self.N,
            device=X_frac.device
        )

        phase = torch.exp(
            1j
            *
            np.pi
            *
            self.alpha
            *
            freqs
        )

        X = X_frac * phase

        recon = torch.fft.ifft(
            X,
            dim=1
        ).real

        return recon

# =====================================================================
# Quantum-Inspired Fractional Domain Filtering
# =====================================================================
# Implements Section 3.3 of the paper.
# Coherent seismic events become concentrated in the fractional domain while incoherent noise
# remains spatially distributed.
# The filtering operation applies adaptive
# sigmoid amplitude gating:
# X̃α[k] = Xα[k] · g(|Xα[k]|)
# where:
# g(z) = sigmoid(β(z - τ))
# Corresponds to:Eq. (15)–(18)
# =====================================================================

class FractionalGate(nn.Module):

    def __init__(self):

        super().__init__()

        self.beta = 8.0

        self.eta = 1.5

    # ================================================================

    def forward(self, X):

        magnitude = torch.abs(X)

        median = torch.median(
            magnitude,
            dim=1,
            keepdim=True
        )[0]

        threshold = self.eta * median

        gain = torch.sigmoid(
            self.beta
            *
            (magnitude - threshold)
        )

        gated = X * gain

        return gated

# =====================================================================
# PENNYLANE QUANTUM DEVICE
# =====================================================================

dev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

# =====================================================================
# Parameterized Quantum Circuit (PQC)
# =====================================================================
# Implements Section 3.4 of the paper.
# The unitary transformation:
# U(θ) = ∏ Ul(θl) consists of:
# - RZ-RY-RZ parameterized rotations
# - CNOT entanglement operations
# These layers extract global seismic correlations
# while preserving signal energy through unitary
# Hilbert-space evolution.
# Corresponds to:Eq. (19)–(24)
# =====================================================================

@qml.qnode(dev, interface="torch")

def pqc_circuit(inputs, weights):

    qml.AmplitudeEmbedding(
        inputs,
        wires=range(N_QUBITS),
        normalize=True
    )

    for layer in range(PQC_LAYERS):

        for qubit in range(N_QUBITS):

            qml.RZ(
                weights[layer, qubit, 0],
                wires=qubit
            )

            qml.RY(
                weights[layer, qubit, 1],
                wires=qubit
            )

            qml.RZ(
                weights[layer, qubit, 2],
                wires=qubit
            )

        for qubit in range(N_QUBITS - 1):

            qml.CNOT(
                wires=[qubit, qubit + 1]
            )

        qml.CNOT(
            wires=[N_QUBITS - 1, 0]
        )

    return qml.state()

# =====================================================================
# PQC LAYER
# =====================================================================

class PQCLayer(nn.Module):

    def __init__(self):

        super().__init__()

        self.weights = nn.Parameter(
            0.005 * torch.randn(
                PQC_LAYERS,
                N_QUBITS,
                3
            )
        )

    # ================================================================

    def forward(self, x):

        outputs = []

        for i in range(x.shape[0]):

            vec = x[i]

            vec = vec / (
                torch.norm(vec) + 1e-8
            )

            state = pqc_circuit(
                vec,
                self.weights
            )

            state_real = torch.real(state)

            outputs.append(state_real)

        outputs = torch.stack(outputs)

        return outputs

# =====================================================================
# Proposed QFrFT-PQCNet Framework
# =====================================================================
# Hybrid quantum-classical seismic denoising model.
# Architecture:
# Stage 1:
#   QFrFT(α1) → Fractional Gating → PQC1
# Stage 2:
#   QFrFT(α2) → Fractional Gating → PQC2
# Final reconstruction is obtained through inverse fractional transformation.
# This follows the cascaded architecture
# =====================================================================

class QFrFTPQCNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.qfrft1 = LearnableQFrFT(N)

        self.qfrft2 = LearnableQFrFT(N)

        self.gate1 = FractionalGate()

        self.gate2 = FractionalGate()

        self.pqc1 = PQCLayer()

        self.pqc2 = PQCLayer()

    # ================================================================

    def forward(self, x):

        # ============================================================
        # Stage 1
        # ============================================================

        frac1 = self.qfrft1(x)

        gated1 = self.gate1(frac1)

        pqc_input1 = torch.abs(gated1)

        pqc_out1 = self.pqc1(pqc_input1)

        # ============================================================
        # Stage 2
        # ============================================================

        frac2 = self.qfrft2(pqc_out1)

        gated2 = self.gate2(frac2)

        pqc_input2 = torch.abs(gated2)

        pqc_out2 = self.pqc2(pqc_input2)

        # ============================================================
        # Reconstruction
        # ============================================================

        recon_complex = torch.complex(
            pqc_out2,
            torch.zeros_like(pqc_out2)
        )

        recon = self.qfrft2.inverse(
            recon_complex
        )

        recon = recon / (
            1.0 + torch.abs(recon)
        )

        return recon

# =====================================================================
# Composite Objective Function
# =====================================================================
# Implements Section 3.6 of the paper.
# Total Objective: L = λ1Ltime + λ2Lfrac + λ3Lcorr
# Ltime :
#     Time-domain reconstruction loss
# Lfrac :
#     Fractional-domain consistency loss
# Lcorr :
#     Correlation preservation loss
#
# The objective simultaneously preserves:
# - waveform fidelity
# - fractional-domain consistency
# - seismic morphology
# Corresponds to:Eq. (29)–(32)
# =====================================================================

class CompositeLoss(nn.Module):

    def __init__(self):

        super().__init__()

    # ================================================================

    def correlation_loss(self, pred, target):

        pred_centered = pred - pred.mean(
            dim=1,
            keepdim=True
        )

        target_centered = target - target.mean(
            dim=1,
            keepdim=True
        )

        numerator = torch.sum(
            pred_centered * target_centered,
            dim=1
        )

        denominator = (
            torch.norm(pred_centered, dim=1)
            *
            torch.norm(target_centered, dim=1)
            + 1e-8
        )

        corr = numerator / denominator

        return 1 - corr.mean()

    # ================================================================

    def forward(self, recon, clean):

        # ============================================================
        # Time-domain reconstruction loss
        # ============================================================

        loss_time = F.smooth_l1_loss(
            recon,
            clean,
            beta=0.5
        )

        # ============================================================
        # Fractional-domain consistency loss
        # ============================================================

        frac_clean = torch.fft.rfft(
            clean,
            dim=1
        )

        frac_recon = torch.fft.rfft(
            recon,
            dim=1
        )

        loss_frac = F.l1_loss(
            torch.log1p(torch.abs(frac_recon)),
            torch.log1p(torch.abs(frac_clean))
        )

        # ============================================================
        # Correlation preservation loss
        # ============================================================

        loss_corr = self.correlation_loss(
            recon,
            clean
        )

        # ============================================================
        # Composite total loss
        # ============================================================

        total_loss = (
            LAMBDA_TIME * loss_time
            +
            LAMBDA_FRAC * loss_frac
            +
            LAMBDA_CORR * loss_corr
        )

        return total_loss

# =====================================================================
# MODEL INITIALIZATION
# =====================================================================

model = QFrFTPQCNet().to(DEVICE)

criterion = CompositeLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=1e-5
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.95
)

# =====================================================================
# Joint Optimization Strategy
# =====================================================================
# Implements Section 3.7 of the paper.
# Jointly optimizes:
# - Fractional transform orders α
# - PQC unitary parameters θ
# Optimization: θ(t+1) = θ(t) - η∇θL
# α(t+1) = α(t) - η∂L/∂α
# using Adam optimizer.
# Stable optimization is achieved using
# - gradient clipping
# - unitary evolution
# - bounded fractional orders
# Corresponds to: Eq. (33)–(37)
# =====================================================================

best_val = 1e9

print("\nStarting Training...\n")

for epoch in range(EPOCHS):

    # ================================================================
    # TRAINING PHASE
    # ================================================================

    model.train()

    running_train_loss = 0.0

    for noisy, clean in train_loader:

        noisy = noisy.to(DEVICE)

        clean = clean.to(DEVICE)

        optimizer.zero_grad()

        recon = model(noisy)

        loss = criterion(
            recon,
            clean
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        running_train_loss += loss.item()

    train_loss = (
        running_train_loss
        /
        len(train_loader)
    )

    # ================================================================
    # VALIDATION PHASE
    # ================================================================

    model.eval()

    running_val_loss = 0.0

    with torch.no_grad():

        for noisy, clean in val_loader:

            noisy = noisy.to(DEVICE)

            clean = clean.to(DEVICE)

            recon = model(noisy)

            loss = criterion(
                recon,
                clean
            )

            running_val_loss += loss.item()

    val_loss = (
        running_val_loss
        /
        len(val_loader)
    )

    scheduler.step()

    # ================================================================
    # MODEL PERFORMANCE
    # ================================================================

    if val_loss < best_val:

        best_val = val_loss

        torch.save(
            model.state_dict(),
            "best_qfrft_pqcnet.pth"
        )

    # ================================================================
    # Print Training Progress Every 5 Epochs
    # ================================================================

    if (epoch + 1) % 5 == 0:

        alpha1 = model.qfrft1.alpha.item()

        alpha2 = model.qfrft2.alpha.item()

        print(
            f"Epoch [{epoch+1:03d}/{EPOCHS}] | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"Alpha1: {alpha1:.4f} | "
            f"Alpha2: {alpha2:.4f}"
        )

# =====================================================================
# TRAINING COMPLETED
# =====================================================================

print("\nTraining Completed Successfully!\n")

# =====================================================================
# Final Learned Fractional Orders
# =====================================================================
# Learned adaptive fractional transform orders after joint optimization.
# These values determine the optimal fractional-domain representation learned  by the framework for seismic denoising.
# =====================================================================
print("Final Learned Fractional Orders")
print("--------------------------------")
print(
    f"Alpha1 = {model.qfrft1.alpha.item():.6f}"
)

print(
    f"Alpha2 = {model.qfrft2.alpha.item():.6f}"
)

# =====================================================================
# END OF IMPLEMENTATION
# =====================================================================

Using Device: cpu

Starting Training...

Epoch [005/100] | Train Loss: 0.633630 | Val Loss: 0.629405 | Alpha1: 0.5000 | Alpha2: 0.4112
Epoch [010/100] | Train Loss: 0.632529 | Val Loss: 0.628568 | Alpha1: 0.5000 | Alpha2: 0.3431
Epoch [015/100] | Train Loss: 0.631972 | Val Loss: 0.628120 | Alpha1: 0.5000 | Alpha2: 0.3076


KeyboardInterrupt: 